In [4]:
import requests
import pandas as pd
import numpy as np
import time
from datetime import timedelta
import random
import yfinance as yf  
from tqdm import tqdm
import os

## Plan


1. Load parquet file with transactions to a dataframe

2. Extract unique tickers

3. Change tickers format to make them compatibles with yahoo finance

4. Use yf to get Sector (possibly market capitalization)

5. Pending: define the dates the model will get inputs (30 - 90 days windows) and fetch historical prices needed

6. Save the data in a csv

7. Later: it must be possible to get new values into the csv and then the code should update the information/append it

## 1. Load parquet file with transactions to a dataframe

In [4]:
df_trans = df = pd.read_parquet("transaction_data/", engine='fastparquet')

In [6]:
df_trans.columns.values

array(['messageId', 'issuerSign', 'publishedTime', 'messageText'],
      dtype=object)

In [14]:
df_tickers = pd.DataFrame(df_trans['issuerSign'].unique(), columns=['Tickers'])
df_tickers['Tickers'] = df_tickers['Tickers'] + '.OL'

In [15]:
df_tickers.head()

,Tickers
0,NORTH.OL
1,2020.OL
2,NOD.OL
3,BWE.OL
4,ZLNA.OL


In [19]:
def get_ticker_details(ticker_symbol):
    # Pause for half a second to be kind to Yahoo's servers
    time.sleep(0.5)
    
    try:
        ticker_obj = yf.Ticker(ticker_symbol)
        info = ticker_obj.info
        
        # Extracting fields directly from the info dictionary
        return pd.Series([
            info.get('currentPrice') or info.get('regularMarketPrice'),
            info.get('marketCap'),
            info.get('currency', 'Unknown'),
            info.get('trailingEps'),
            info.get('trailingPE'),
            info.get('sector', 'N/A')
        ])
    
    except Exception:
        # Return Nones if the ticker fails
        return pd.Series([None, None, "Error", None, None, "N/A"])


In [20]:
# Apply the function to your DataFrame
# Make sure your column names match the order in the Series above
tqdm.pandas()
df_tickers[['Price', 'Market Cap', 'Currency', 'EPS', 'PE Ratio', 'Sector']] = df_tickers['Tickers'].apply(get_ticker_details)



Failed to perform, curl: (16) . See https://curl.se/libcurl/c/libcurl-errors.html first for more details.
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: BFISH.OL"}}}
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: AMSC.OL"}}}
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: CSS.OL"}}}
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: AIDR.OL"}}}
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: ARGEO.OL"}}}
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: DFDS.OL"}}}
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for sym

   messageId issuerSign             publishedTime  \
0     608085      NORTH  2024-01-05T15:08:29.699Z   
1     608082       2020  2024-01-05T15:03:05.974Z   
2     608040        NOD  2024-01-05T08:00:00.000Z   
3     608027        BWE  2024-01-05T06:30:00.601Z   
4     608023       ZLNA  2024-01-04T17:57:24.502Z   

                                         messageText  
0  5.1.2024 16:08:26 CET | North Energy ASA | Man...  
1  Kate Blankenship, Director of 2020 Bulkers Ltd...  
2  (Oslo, January 5, 2024) Grant of Restricted St...  
3  Mandatory notifications of trade by primary in...  
4  Oslo, 4 January 2024: Carlos de Sousa, Chief E...  


In [21]:
df_tickers.head()


,Tickers,Price,Market Cap,Currency,EPS,PE Ratio,Sector
0,NORTH.OL,2.70,3.165793e+08,NOK,-0.34,None,Financial Services
1,2020.OL,140.40,3.219499e+09,NOK,12.57,11.169451,Industrials
2,NOD.OL,152.70,3.016390e+10,NOK,0.78,195.76924,Technology
3,BWE.OL,58.30,1.505710e+10,NOK,4.97,11.730383,Energy
4,ZLNA.OL,18.78,4.933468e+08,NOK,-7.10,None,Healthcare


In [22]:
df_tickers.to_csv('oslo_stock_data.csv', index=False)
print("File saved successfully!")

File saved successfully!


In [8]:
# MÅ RYDDE DISSE HER

def get_ticker_details(ticker_symbol):
    """Fetches financial data for a single ticker."""
    time.sleep(0.5) # Courtesy delay
    try:
        ticker_obj = yf.Ticker(ticker_symbol)
        info = ticker_obj.info
        return [
            info.get('currentPrice') or info.get('regularMarketPrice'),
            info.get('marketCap'),
            info.get('currency', 'Unknown'),
            info.get('trailingEps'),
            info.get('trailingPE'),
            info.get('sector', 'N/A')
        ]
    except Exception:
        return [None, None, "Error", None, None, "N/A"]

def update_master_with_new_tickers(df_master, parquet_path):
    """
    Identifies new tickers in transactions, fetches their data, 
    and appends them to the master DataFrame.
    """
    # 1. Internal extraction of unique tickers from parquet
    df_tx = pd.read_parquet(parquet_path, engine='fastparquet')
    df_tx = pd.DataFrame(df_tx['issuerSign'].unique(), columns=['Tickers'])
    df_tx['Tickers'] = df_tx['Tickers'] + '.OL'
    unique_tx_tickers = df_tx['Tickers'].unique()

    # 2. Compare and find missing tickers
    existing_tickers = set(df_master['Tickers'])
    new_tickers = [t for t in unique_tx_tickers if t not in existing_tickers]

    if not new_tickers:
        print("No new tickers found. Everything is up to date.")
        return df_master

    print(f"Found {len(new_tickers)} new tickers. Fetching data...")

    # 3. Fetch and build new rows
    new_rows = []
    for ticker in new_tickers:
        print(f"Processing: {ticker}")
        details = get_ticker_details(ticker)
        new_rows.append([ticker] + details)

    # 4. Append to master
    df_new_entries = pd.DataFrame(new_rows, columns=df_master.columns)
    df_updated = pd.concat([df_master, df_new_entries], ignore_index=True)
    
    return df_updated



In [9]:
# --- MAIN EXECUTION FLOW ---

MASTER_FILE = 'oslo_stock_data.csv'
PARQUET_FILE = 'transaction_data/'

# Load (or create) the Master DataFrame
if os.path.exists(MASTER_FILE):
    master_df = pd.read_csv(MASTER_FILE)
else:
    cols = ['Tickers', 'Price', 'Market Cap', 'Currency', 'EPS', 'PE Ratio', 'Sector']
    master_df = pd.DataFrame(columns=cols)

# Run the update
master_df = update_master_with_new_tickers(master_df, PARQUET_FILE)

# Save the results back to the CSV
master_df.to_csv(MASTER_FILE, index=False, encoding='utf-8-sig')
print("Workflow complete.")

No new tickers found. Everything is up to date.
Workflow complete.
